# multixkv  —  Sec 5.3  sender-pool scaling to N=8

Frozen Qwen2.5-1.5B, split-context SQuAD. Anchors are re-measured
in-run (floor = reader's own masked cache, reference = its unmasked cache).

Results are written to `results/`, figures to `figures/`.


# Multi-agent same-family sharing — verbatim-KV augmentation over a POOL of holders

The XKV-LoRA positive (`same:qkv` shared **0.695** > floor **0.18**, ctrl clean) used **one**
holder that held the reader's own passage un-masked. That is the cleanest sharing test but not yet a
*multi-agent* one. Here the reader cross-attends the **union of N distinct holders' verbatim KV**, exactly
**one** of which carries the answer; the rest are distractors. Only the reader's own `W_q/W_k/W_v` train
(hand-rolled LoRA, `qkv` arm). Holder KV is verbatim — substitution is never used (the oracle killed it).

$$\text{ctx}=\mathrm{softmax}\!\left(\tfrac{q'\,[K'_{self};K_{h_1};\dots;K_{h_N}]^\top}{\sqrt d}\right)
[V'_{self};V_{h_1};\dots;V_{h_N}]$$

**Axes:** `N_HOLDERS ∈ {1,2,4,8}` × arms `{raw, qkv}`.
**Conditions:** floor / ceiling / shared / ctrl_masked (answer-holder masked) / ctrl_distract (no answer in pool).
**Success:** `shared > floor` at `N>1` with BOTH controls at floor and no cliff as `N` grows.

In [ ]:
import os
# Write next to this notebook's parent (the repo root), not the CWD, so
# results land in the same results/ that make_figs.py and verify_results.py read.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)
import os
CACHE = os.environ.get("HF_HOME") or os.path.expanduser("~/.cache/huggingface")
os.environ["HF_HOME"]=CACHE; os.environ["HF_DATASETS_CACHE"]=CACHE
os.environ["TOKENIZERS_PARALLELISM"]="false"
import json, numpy as np, re
import torch, torch.nn as nn, torch.nn.functional as F
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
try:
    get_ipython().run_line_magic("matplotlib","inline")
except NameError:
    pass  # running as a plain script, not under IPython
assert torch.cuda.is_available(), (
    "This experiment needs a CUDA GPU. The reader runs in fp32 (deliberately: the\n"
    "cosine and attention-logit measurements are sensitive to reduced precision),\n"
    "so expect ~8 GB and more for the larger notebooks.")
DEVICE="cuda"; SEED=0; torch.manual_seed(SEED); np.random.seed(SEED)
print("Device:",torch.cuda.get_device_name(0),flush=True)
BASE="Qwen/Qwen2.5-1.5B"
CFG=dict(
  lora_R=16, bridge_epochs=4, bridge_lr=2e-4,
  n_train=200, n_eval=200, max_ctx=256, gen_len=8,
  arms=["raw","qkv"], n_holders=[1,2,4,8], keepbest_slice=32,
  train_N=4,                 # train the bridge at a fixed pool size (mid of the sweep)
)
if os.environ.get("MXKV_FAST", os.environ.get("KVL_FAST","0"))=="1":
    CFG.update(n_train=24,n_eval=24,bridge_epochs=2,keepbest_slice=12,n_holders=[1,2,4])
    print(">>> FAST <<<",flush=True)
# A fast run writes to a separate tag so it cannot overwrite the
# committed paper results.
_default_tag = "_multixkvfast" if os.environ.get("MXKV_FAST", os.environ.get("KVL_FAST","0"))=="1" else "_multixkv"
TAG=os.environ.get("MXKV_TAG", os.environ.get("KVL_TAG", None)) or ( _default_tag)
print("CFG",CFG,"TAG",TAG,flush=True)

## 1. Load frozen reader (Qwen); grab W_Q/W_K/W_V (+biases), RoPE  — identical to XKV-LoRA

In [ ]:
tok=AutoTokenizer.from_pretrained(BASE,cache_dir=CACHE)
if tok.pad_token is None: tok.pad_token=tok.eos_token
base=AutoModelForCausalLM.from_pretrained(BASE,cache_dir=CACHE,torch_dtype=torch.float32).to(DEVICE).eval()
for p in base.parameters(): p.requires_grad_(False)
ids=tok("hi",return_tensors="pt").input_ids.to(DEVICE)
with torch.no_grad(): pkv=base(ids,use_cache=True).past_key_values
if not isinstance(pkv,tuple): pkv=pkv.to_legacy_cache() if hasattr(pkv,"to_legacy_cache") else pkv
N_LAYER=len(pkv); _,N_KV,_,D_HEAD=pkv[0][0].shape
N_Q=base.config.num_attention_heads; GROUP=N_Q//N_KV; KV_DIM=2*N_KV*D_HEAD
KDIM=N_KV*D_HEAD; HID=base.config.hidden_size
layers=base.model.layers
WQ=[layers[l].self_attn.q_proj.weight.detach() for l in range(N_LAYER)]
BQ=[getattr(layers[l].self_attn.q_proj,'bias',None) for l in range(N_LAYER)]
WK=[layers[l].self_attn.k_proj.weight.detach() for l in range(N_LAYER)]
BK=[getattr(layers[l].self_attn.k_proj,'bias',None) for l in range(N_LAYER)]
WV=[layers[l].self_attn.v_proj.weight.detach() for l in range(N_LAYER)]
BV=[getattr(layers[l].self_attn.v_proj,'bias',None) for l in range(N_LAYER)]
QDIM=N_Q*D_HEAD
print(f"L={N_LAYER} n_q={N_Q} n_kv={N_KV} group={GROUP} d_head={D_HEAD} kdim={KDIM} qdim={QDIM} hid={HID}",flush=True)
_THETA=float(getattr(base.config,'rope_theta',1e6))
_INV=1.0/(_THETA**(torch.arange(0,D_HEAD,2,device=DEVICE).float()/D_HEAD))
def rope_at(x,positions):
    ang=torch.outer(positions.float(),_INV); emb=torch.cat([ang,ang],-1)
    cos=emb.cos(); sin=emb.sin()
    t1,t2=x[...,:D_HEAD//2],x[...,D_HEAD//2:]; rr=torch.cat([-t2,t1],-1)
    return x*cos+rr*sin
def _heads(x,nh):  # (T, nh*dh) -> (nh,T,dh)
    T=x.shape[0]; return x.view(T,nh,D_HEAD).permute(1,0,2)
def _unheads(x):   # (nh,T,dh) -> (T, nh*dh)
    nh,T,dh=x.shape; return x.permute(1,0,2).reshape(T,nh*dh)

## 2. Cache tooling; split-context SQuAD; capture reader hidden — identical to XKV-LoRA

In [ ]:
def kv_stack(past):
    outs=[]
    for (k,v) in past:
        B,H,T,dh=k.shape
        outs.append(torch.cat([k.permute(0,2,1,3).reshape(B,T,H*dh),
                               v.permute(0,2,1,3).reshape(B,T,H*dh)],-1))
    return torch.stack(outs,0)
def native_ids(text,max_len,tk=None):
    tk=tk or tok
    return tk([text],return_tensors="pt",truncation=True,max_length=max_len).input_ids.to(DEVICE)
@torch.no_grad()
def reader_cache(text,max_len):
    ids=native_ids(text,max_len); last=ids[:,-1:]; body=ids[:,:-1]
    leg=base(body,use_cache=True).past_key_values
    if not isinstance(leg,tuple): leg=leg.to_legacy_cache() if hasattr(leg,"to_legacy_cache") else leg
    return kv_stack(leg), last, body
@torch.no_grad()
def capture_h(body):
    Hs={}; hooks=[]
    def mk(l):
        def h(mod,inp,out): Hs[l]=out[0].detach()   # (T,HID)
        return h
    for l in range(N_LAYER): hooks.append(layers[l].input_layernorm.register_forward_hook(mk(l)))
    base(body,use_cache=False)
    for h in hooks: h.remove()
    return Hs

def make_prompt(p,q): return f"{p}\nQuestion: {q}\nAnswer:"
def mask_answer(p,a):
    i=p.find(a); return None if i<0 else p[:i]+"____"+p[i+len(a):]
def build_qa(n,split):
    ds=datasets.load_dataset("squad",split=split,cache_dir=CACHE); items=[]
    for ex in ds:
        a=ex["answers"]["text"][0].strip()
        if not a or a not in ex["context"] or len(a.split())>4: continue
        m=mask_answer(ex["context"],a)
        if m is None: continue
        items.append(dict(q=ex["question"].strip(),full=ex["context"].strip(),masked=m.strip(),ans=a))
        if len(items)>=n: break
    return items
train_qa=build_qa(CFG["n_train"],"train"); eval_qa=build_qa(CFG["n_eval"],"validation")
# SQuAD loads grouped by article (all Super Bowl 50 first, etc.) -> sequential distractors are same-topic and
# weak, and can share answers. Shuffle deterministically so a pool spans topics = a fair distractor test.
def _shuffle(xs):
    g=torch.Generator().manual_seed(SEED); return [xs[k] for k in torch.randperm(len(xs),generator=g).tolist()]
train_qa=_shuffle(train_qa); eval_qa=_shuffle(eval_qa)
print("train",len(train_qa),"eval",len(eval_qa),flush=True)
def norm(s): return re.sub(r"[^a-z0-9 ]","",s.lower()).strip()
def hit(t,g): return norm(g) in norm(t) if g else False

## 3. Holder caches (same-family) and POOL builders

Each holder cache is the reader's own verbatim KV over some passage (same geometry — this is the live
branch). A POOL for item `it` at size `N` = the answer-holder (that item's full un-masked context) plus
`N-1` DISTRACTOR holders drawn from OTHER eval items (their full contexts, none containing this item's
answer). We concatenate holders along the token axis: `[K_h1;...;K_hN]`. Because holders are independent
caches, each is RoPE'd at its own local positions `[0..T_h)` when built (via `qwen_holder_kv`, which reuses
`reader_cache` — post-RoPE keys). The answer-holder's SLOT in the pool is randomized per item so the reader
cannot learn a positional shortcut.

In [ ]:
@torch.no_grad()
def qwen_holder_kv(text):
    vec,_,_=reader_cache(text,CFG["max_ctx"]); return vec   # (L,1,T,KV_DIM) reader-geometry, POST-RoPE keys

def _distractor_pool(items):
    # precompute a shuffled index ring of distractor sources (their FULL contexts)
    idx=list(range(len(items)))
    g=torch.Generator().manual_seed(SEED); perm=torch.randperm(len(items),generator=g).tolist()
    return perm

def concat_holders(vec_list):
    # vec_list: list of (L,1,T_i,KV_DIM) -> (L,1,sum T_i, KV_DIM); empty -> zero-length
    if len(vec_list)==0:
        return torch.zeros(N_LAYER,1,0,KV_DIM,device=DEVICE)
    return torch.cat(vec_list,dim=2)

def build_pool(it, i, items, ring, N, mode):
    # mode: 'shared' (one answer-holder + N-1 distractors), 'ctrl_masked' (answer-holder MASKED + distractors),
    #       'ctrl_distract' (N distractors, NO answer-holder). Deterministic per (i,N,mode).
    # `ring` is IGNORED (kept for signature stability): distractor indices are derived here from len(items) so
    # they ALWAYS index the list actually passed (eval_shared_slice may pass a shorter slice than the ring's
    # source list — that mismatch was the IndexError). Deterministic stride over items != i.
    M=len(items)
    need = N if mode=="ctrl_distract" else N-1
    ds=[]; k=1
    stride=7   # coprime-ish walk so distractors vary with i without RNG (avoids RNG so the pool is deterministic)
    while len(ds)<need and k<=M:
        j=(i + k*stride) % M; k+=1
        if j==i: continue
        # Reject a candidate ONLY if it genuinely answers this item's question — approximated by same answer
        # string. A raw substring test ("ans appears anywhere in ctx") rejected 100% of SQuAD contexts (common
        # tokens like "gold"/years/team names recur across passages) and STARVED the pool to the answer-holder
        # alone (fast smoke: shared frozen at the N=1 value across all N). Same-answer match is the right guard.
        if items[j]["ans"] and norm(items[j]["ans"])==norm(it["ans"]): continue
        if j in ds: continue
        ds.append(j)
    assert len(ds)==need, f"pool starved: item {i} got {len(ds)}/{need} distractors (screen too aggressive?)"
    dist=[qwen_holder_kv(make_prompt(items[j]["full"],items[j]["q"])) for j in ds]
    if mode=="ctrl_distract":
        holders=dist
    else:
        ans_text = make_prompt(it["masked"] if mode=="ctrl_masked" else it["full"], it["q"])
        ans_h=qwen_holder_kv(ans_text)
        # deterministic slot for the answer-holder among the actual pool positions
        slot=(i*7+N)%(len(dist)+1)
        holders=dist[:]; holders.insert(slot,ans_h)
    return concat_holders(holders)

## 4. LoRAKQV + manual reader forward with MULTI-holder cross-attention

Identical to XKV-LoRA except the holder axis is now the concatenated pool. `xkv_attend` already handles an
arbitrary-length holder block (`T_holder` = total pooled tokens); only `V_holder`/`K_holder` grow.

In [ ]:
class LoRAKQV(nn.Module):
    def __init__(s,arm):
        super().__init__(); s.arm=arm; R=CFG["lora_R"]
        s.use_q = arm in ("qonly","qkv")
        s.use_k = arm in ("kv_only","qkv")
        s.use_v = arm in ("kv_only","qkv")
        def mk(out):
            A=nn.ParameterList([nn.Parameter(0.01*torch.randn(HID,R)) for _ in range(N_LAYER)])
            B=nn.ParameterList([nn.Parameter(torch.zeros(R,out)) for _ in range(N_LAYER)])  # zero -> identity
            return A,B
        if s.use_q: s.Aq,s.Bq=mk(QDIM)
        if s.use_k: s.Ak,s.Bk=mk(KDIM)
        if s.use_v: s.Av,s.Bv=mk(KDIM)
    def dq(s,l,h): return (h@s.Aq[l])@s.Bq[l] if s.use_q else 0.0
    def dk(s,l,h): return (h@s.Ak[l])@s.Bk[l] if s.use_k else 0.0
    def dv(s,l,h): return (h@s.Av[l])@s.Bv[l] if s.use_v else 0.0

SCALE=1.0/(D_HEAD**0.5)
def build_self_kv(lora, l, h_prefix, positions):
    kflat=h_prefix@WK[l].T + (BK[l] if BK[l] is not None else 0.0) + lora.dk(l,h_prefix)
    vflat=h_prefix@WV[l].T + (BV[l] if BV[l] is not None else 0.0) + lora.dv(l,h_prefix)
    kh=_heads(kflat,N_KV); kh=rope_at(kh,positions); kflat=_unheads(kh)
    return torch.cat([kflat,vflat],-1)

def xkv_attend(lora, h_layer_in, l, self_KV, holder_KV, pos_query, T_holder):
    ln=layers[l].input_layernorm(h_layer_in).reshape(1,HID)
    qflat=ln@WQ[l].T + (BQ[l] if BQ[l] is not None else 0.0) + lora.dq(l,ln)
    q=_heads(qflat,N_Q); q=rope_at(q, torch.tensor([pos_query],device=DEVICE))
    Ks=_heads(self_KV[:,:KDIM],N_KV); Vs=_heads(self_KV[:,KDIM:],N_KV)
    Ks_e=Ks.repeat_interleave(GROUP,0); Vs_e=Vs.repeat_interleave(GROUP,0)
    if T_holder>0:
        Kh=_heads(holder_KV[:,:KDIM],N_KV); Vh=_heads(holder_KV[:,KDIM:],N_KV)
        Kall=torch.cat([Ks_e,Kh.repeat_interleave(GROUP,0)],1)
        Vall=torch.cat([Vs_e,Vh.repeat_interleave(GROUP,0)],1)
    else:
        Kall=Ks_e; Vall=Vs_e
    att=torch.einsum('hqd,hkd->hqk', q, Kall)*SCALE
    w=torch.softmax(att,dim=-1)
    ctx=torch.einsum('hqk,hkd->hqd', w, Vall).permute(1,0,2).reshape(1,QDIM)
    return layers[l].self_attn.o_proj(ctx).view(1,1,HID)

def reader_block_forward(l, h_in, attn_out):
    h=h_in+attn_out; ln2=layers[l].post_attention_layernorm(h); return h+layers[l].mlp(ln2)

def mxkv_generate(lora, Hs_prefix, last_id, pool_vec):
    # pool_vec:(L,1,T_pool,KV_DIM) concatenated holders. Same as xkv_generate with a longer holder block.
    T_self=Hs_prefix[0].shape[0]; T_pool=pool_vec.shape[2]
    pos_prefix=torch.arange(T_self,device=DEVICE)
    selfKV=[build_self_kv(lora,l,Hs_prefix[l],pos_prefix) for l in range(N_LAYER)]
    cur=last_id; gen=[]; pos=T_self
    for step in range(CFG["gen_len"]):
        h=base.model.embed_tokens(cur)
        for l in range(N_LAYER):
            ln=layers[l].input_layernorm(h).reshape(1,HID)
            k=ln@WK[l].T + (BK[l] if BK[l] is not None else 0.0) + lora.dk(l,ln)
            v=ln@WV[l].T + (BV[l] if BV[l] is not None else 0.0) + lora.dv(l,ln)
            kh=_heads(k,N_KV); kh=rope_at(kh, torch.tensor([pos],device=DEVICE)); k=_unheads(kh)
            selfKV[l]=torch.cat([selfKV[l], torch.cat([k,v],-1)],0)
            attn=xkv_attend(lora,h,l,selfKV[l],pool_vec[l,0],pos,T_pool)
            h=reader_block_forward(l,h,attn)
        h=base.model.norm(h); logit=base.lm_head(h)[:, -1, :]
        nxt=logit.argmax(-1,keepdim=True); gen.append(nxt); cur=nxt; pos+=1
    return tok.decode(torch.cat(gen,1)[0])

## 4c. PARITY GATES — manual==native, rebuilt-raw==native (reused), + N=1 single-holder sanity

In [ ]:
from transformers.cache_utils import DynamicCache
def slots_to_legacy(vec):
    L,B,Kk,_=vec.shape; half=KDIM; leg=[]
    for l in range(L):
        kt=vec[l,:,:,:half].reshape(B,Kk,N_KV,D_HEAD).permute(0,2,1,3).contiguous()
        vt=vec[l,:,:,half:].reshape(B,Kk,N_KV,D_HEAD).permute(0,2,1,3).contiguous()
        leg.append((kt,vt))
    return leg
@torch.no_grad()
def native_greedy(prefix_vec,last_id):
    past=DynamicCache.from_legacy_cache(tuple(slots_to_legacy(prefix_vec))); T0=prefix_vec.shape[2]
    def pos(s,l): return torch.arange(s,s+l,device=DEVICE).unsqueeze(0)
    o=base(last_id,past_key_values=past,position_ids=pos(T0,1),
           attention_mask=torch.ones(1,T0+1,device=DEVICE),use_cache=True)
    past=o.past_key_values; cur=o.logits[:,-1,:].argmax(-1,keepdim=True); gen=[cur]; L=T0+1
    for _ in range(CFG["gen_len"]-1):
        o=base(cur,past_key_values=past,position_ids=pos(L,1),
               attention_mask=torch.ones(1,L+1,device=DEVICE),use_cache=True)
        past=o.past_key_values; cur=o.logits[:,-1,:].argmax(-1,keepdim=True); gen.append(cur); L+=1
    return tok.decode(torch.cat(gen,1)[0])

_raw=LoRAKQV("raw").to(DEVICE); _mx=0.0
for it in eval_qa[:min(4,len(eval_qa))]:
    vec,last,body=reader_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"])
    Hs=capture_h(body); T=vec.shape[2]; posp=torch.arange(T,device=DEVICE)
    for l in range(N_LAYER):
        reb=build_self_kv(_raw,l,Hs[l],posp)
        _mx=max(_mx, float((reb-vec[l,0]).abs().max()))
print(f"REBUILT-RAW vs native cache maxabs diff: {_mx:.3e}",flush=True)
assert _mx<1e-2, "rebuilt prefix K/V diverges from native cache"

_match=0; _tot=0
for it in eval_qa[:min(8,len(eval_qa))]:
    vec,last,body=reader_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"])
    Hs=capture_h(body)
    man=mxkv_generate(_raw,Hs,last,vec[:, :, :0, :])
    nat=native_greedy(vec,last)
    _tot+=1; _match+= (man.strip()==nat.strip())
print(f"PARITY manual==native: {_match}/{_tot}",flush=True)
assert _match>=_tot-1, "manual forward diverges from native"

# N=1 single-holder sanity: pool = {answer-holder only} with raw LoRA must reproduce the single-sender identity
# regime (masked self + own-full holder). Just report it — a positive here anchors against that experiment.
ring=_distractor_pool(eval_qa)
_id=0; _n=0
for i,it in enumerate(eval_qa[:min(20,len(eval_qa))]):
    vec,last,body=reader_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"]); Hs=capture_h(body)
    pool=build_pool(it,i,eval_qa,ring,1,"shared")   # N=1 shared == single answer-holder
    _id+=hit(mxkv_generate(_raw,Hs,last,pool),it["ans"]); _n+=1
print(f"N=1 raw single-holder shared (single-sender reference 0.555): {_id/_n:.3f} on {_n}",flush=True)
print("parity OK",flush=True)

## 5. Train the qkv bridge at a fixed pool size (train_N) with KEEP-BEST guard vs raw

In [ ]:
_ring_tr=_distractor_pool(train_qa)

def ce_loss_mxkv(lora, it, i, items, ring, N):
    vec,last,body=reader_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"])
    Hs=capture_h(body)
    pool=build_pool(it,i,items,ring,N,"shared")
    gold=tok(" "+it["ans"],return_tensors="pt").input_ids.to(DEVICE)[:, :1] if it["ans"] else None
    if gold is None: return None
    T_self=Hs[0].shape[0]; T_pool=pool.shape[2]; pos=T_self
    posp=torch.arange(T_self,device=DEVICE)
    selfKV=[build_self_kv(lora,l,Hs[l],posp) for l in range(N_LAYER)]
    h=base.model.embed_tokens(last)
    for l in range(N_LAYER):
        attn=xkv_attend(lora,h,l,selfKV[l],pool[l,0],pos,T_pool)
        h=reader_block_forward(l,h,attn)
    h=base.model.norm(h); logit=base.lm_head(h)[:, -1, :]
    return F.cross_entropy(logit, gold.view(-1))

@torch.no_grad()
def eval_shared_slice(lora,items,ring,N):
    n=0; hits=0
    for i,it in enumerate(items):
        vec,last,body=reader_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"])
        Hs=capture_h(body); pool=build_pool(it,i,items,ring,N,"shared")
        hits+=hit(mxkv_generate(lora,Hs,last,pool),it["ans"]); n+=1
    return hits/max(1,n)

def train_lora(arm,data,ring,N):
    lora=LoRAKQV(arm).to(DEVICE)
    if arm=="raw": return lora
    slice_items=data[:CFG["keepbest_slice"]]
    raw0=eval_shared_slice(LoRAKQV("raw").to(DEVICE),slice_items,ring,N)
    ep0=eval_shared_slice(lora,slice_items,ring,N)
    assert abs(ep0-raw0)<1e-9, f"{arm} untrained != raw (raw0={raw0} ep0={ep0})"
    best_state={k:v.detach().clone() for k,v in lora.state_dict().items()}; best=max(raw0,ep0)
    print(f"  [{arm} N={N}] ep0 slice_shared={ep0:.3f} (raw={raw0:.3f})",flush=True)
    opt=torch.optim.Adam([p for p in lora.parameters()],lr=CFG["bridge_lr"])
    for ep in range(CFG["bridge_epochs"]):
        idx=torch.randperm(len(data)).tolist(); tot=0.0; n=0
        for j in idx:
            L=ce_loss_mxkv(lora,data[j],j,data,ring,N)
            if L is None or not torch.isfinite(L): continue
            opt.zero_grad(); L.backward(); torch.nn.utils.clip_grad_norm_(lora.parameters(),1.0); opt.step()
            tot+=float(L); n+=1
        sh=eval_shared_slice(lora,slice_items,ring,N)
        keep=""
        if sh>best: best=sh; best_state={k:v.detach().clone() for k,v in lora.state_dict().items()}; keep=" <-keep"
        print(f"  [{arm} N={N}] ep{ep+1} ce={tot/max(1,n):.4f} slice_shared={sh:.3f} best={best:.3f}{keep}",flush=True)
    lora.load_state_dict(best_state)
    return lora

# One trained qkv bridge (trained at pool size train_N), evaluated across ALL N. raw is the parity anchor.
TRAINED={"raw":LoRAKQV("raw").to(DEVICE)}
if "qkv" in CFG["arms"]:
    TRAINED["qkv"]=train_lora("qkv",train_qa,_ring_tr,CFG["train_N"])

## 6. Eval across pool sizes N — floor / ceiling / shared / ctrl_masked / ctrl_distract

In [ ]:
_ring_ev=_distractor_pool(eval_qa)

@torch.no_grad()
def eval_at_N(lora, items, ring, N, want_ceiling):
    A={"floor":0,"ceiling":0,"shared":0,"ctrl_masked":0,"ctrl_distract":0}; n=0
    raw=TRAINED["raw"]
    for i,it in enumerate(items):
        g=it["ans"]
        vec,last,body=reader_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"]); Hs=capture_h(body)
        empty=vec[:, :, :0, :]
        A["floor"]+=hit(mxkv_generate(raw,Hs,last,empty),g)
        if want_ceiling:
            fvec,lastf,fbody=reader_cache(make_prompt(it["full"],it["q"]),CFG["max_ctx"]); Hf=capture_h(fbody)
            A["ceiling"]+=hit(mxkv_generate(raw,Hf,lastf,empty),g)
        A["shared"]       +=hit(mxkv_generate(lora,Hs,last,build_pool(it,i,items,ring,N,"shared")),g)
        A["ctrl_masked"]  +=hit(mxkv_generate(lora,Hs,last,build_pool(it,i,items,ring,N,"ctrl_masked")),g)
        A["ctrl_distract"]+=hit(mxkv_generate(lora,Hs,last,build_pool(it,i,items,ring,N,"ctrl_distract")),g)
        n+=1
    return {k:v/max(1,n) for k,v in A.items()}

RES={"config":CFG,"base":BASE,"runs":{}}
RESPATH=f"results/multixkv{TAG}.json"
CEIL=None
for arm in CFG["arms"]:
    lora=TRAINED[arm]
    for ni,N in enumerate(CFG["n_holders"]):
        want=(arm=="raw" and ni==0)
        m=eval_at_N(lora,eval_qa,_ring_ev,N,want_ceiling=want)
        if want: CEIL=m["ceiling"]
        else:    m["ceiling"]=CEIL if CEIL is not None else m["ceiling"]   # broadcast the single real ceiling
        RES["runs"][f"{arm}:N{N}"]=m
        print(f"[{arm} N={N}] floor={m['floor']:.3f} ceil={m['ceiling']:.3f} shared={m['shared']:.3f} "
              f"ctrl_masked={m['ctrl_masked']:.3f} ctrl_distract={m['ctrl_distract']:.3f}",flush=True)
        json.dump(RES,open(RESPATH,"w"),indent=1)
json.dump(RES,open(RESPATH,"w"),indent=1); print("saved",RESPATH,flush=True)

## 7. Verdict

In [ ]:
fl=RES["runs"][f"raw:N{CFG['n_holders'][0]}"]["floor"]
ce=RES["runs"][f"raw:N{CFG['n_holders'][0]}"]["ceiling"]
print(f"\n=== MULTI-AGENT SHARING VERDICT — floor={fl:.3f} ceiling={ce:.3f} ===",flush=True)
for arm in CFG["arms"]:
    print(f" [{arm}]",flush=True)
    for N in CFG["n_holders"]:
        m=RES["runs"][f"{arm}:N{N}"]
        sh=m["shared"]; cm=m["ctrl_masked"]; cd=m["ctrl_distract"]
        beat=" <-- BEATS FLOOR" if sh>fl+1e-9 else ""
        ctrl_clean = (cm<=fl+0.03 and cd<=fl+0.03)
        tag=" (ctrls clean)" if ctrl_clean else " (a CTRL lifts - suspect)"
        print(f"   N={N}: shared={sh:.3f} ctrl_masked={cm:.3f} ctrl_distract={cd:.3f}{tag}{beat}",flush=True)
print("\nSuccess = shared>floor at N>1 with BOTH controls at floor and no cliff as N grows.",flush=True)

## 8. Plot — shared vs controls across pool size N

In [ ]:
fig,ax=plt.subplots(1,len(CFG["arms"]),figsize=(6.4*len(CFG["arms"]),4.4),squeeze=False)
Ns=CFG["n_holders"]; x=np.arange(len(Ns))
fl=RES["runs"][f"raw:N{Ns[0]}"]["floor"]; ce=RES["runs"][f"raw:N{Ns[0]}"]["ceiling"]
for ai,arm in enumerate(CFG["arms"]):
    a=ax[0][ai]
    sh=[RES["runs"][f"{arm}:N{N}"]["shared"] for N in Ns]
    cm=[RES["runs"][f"{arm}:N{N}"]["ctrl_masked"] for N in Ns]
    cd=[RES["runs"][f"{arm}:N{N}"]["ctrl_distract"] for N in Ns]
    a.plot(x,sh,'-o',color="#2c7fb8",label="shared (answer in pool)")
    a.plot(x,cm,'--s',color="#d95f0e",label="ctrl_masked (answer blanked)")
    a.plot(x,cd,'--^',color="#e6a000",label="ctrl_distract (no answer)")
    a.axhline(fl,ls=':',color='#999',label=f"floor {fl:.2f}"); a.axhline(ce,ls='-',color='#000',label=f"ceil {ce:.2f}")
    a.set_xticks(x); a.set_xticklabels([f"N={n}" for n in Ns]); a.set_ylim(0,max(ce,max(sh))+0.08)
    a.set_title(f"Multi-agent sharing — {arm}"); a.legend(fontsize=7); a.grid(ls=':',alpha=.4)
    a.set_xlabel("pool size (1 answer-holder + N-1 distractors)")
fig.tight_layout(); fig.savefig("figures/multixkv_curves.png",dpi=150); plt.show()